# <center> Word2Vec </center>


## <center> Data Analysis </center>

In [2]:
import numpy as np
import pandas as pd
import os


# Путь к данным
train_path = '../aclImdb/train'

# Функция для чтения данных из файлов
def read_data(folder, label, base_path):
    data = []
    folder_path = os.path.join(base_path, folder)
    
    for filename in os.listdir(folder_path):
        if filename.endswith('.txt'):
            # Чтение текста из файла
            with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as file:
                text = file.read()
            
            # Извлечение target из названия файла
            target = filename.split('_')[1].split('.')[0]
            
            # Добавление данных в список
            data.append({
                'review': text,
                'sentiment': label
            })
    
    return data

In [3]:

# Чтение данных из папок pos и neg
pos_data = read_data('pos', 1, train_path)
neg_data = read_data('neg', 0, train_path)


test_path = '../aclImdb/test'
pos_data_test = read_data('pos', 1, test_path)
neg_data_test = read_data('neg', 0, test_path)


# Объединение данных
all_data = pos_data + neg_data
test_data = pos_data_test + neg_data_test

# Создание DataFrame
df = pd.DataFrame(all_data)
df_test = pd.DataFrame(test_data)


df

,review,sentiment
0,"Another demonstration of Kurosawa's genius, hi...",1
1,A new side to the story of Victoria and Albert...,1
2,Whoever says pokemon is stupid can die. This m...,1
3,"With Iphigenia, Mikhali Cacoyannis is perhaps ...",1
4,I have to start saying it has been a long time...,1
...,...,...
24995,The movie was TERRIBLE!!! Easily the worst mov...,0
24996,"this movie has no plot, no character developme...",0
24997,I don't know what it is with these Brady kids....,0
24998,"Firstly, there are some good things about this...",0


In [4]:
#Details about the columns of the Dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     25000 non-null  object
 1   sentiment  25000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 390.8+ KB


In [5]:
#Finding out total number of null values in either columns
df.isnull().sum()

review       0
sentiment    0
dtype: int64

# <center> Data Cleaning </center>

In [6]:
!pip install contractions

In [7]:
import contractions
#This package is used to replace the contractions in English language with their actual forms
from tqdm import tqdm
#tqdm is used to display the percentage of work done by a for loop.
import nltk
#Contains various language specific datasets and tools for analysis
import re
#used to work with regular expression and helps in finding the matches for a given regex.
import time
nltk.download('stopwords')
from nltk.corpus import stopwords
#donwloadin the stopwords of english language
stopwords=stopwords.words('english')
#Removing stopwords 'no','nor' and 'not'
stopwords.remove('no')
stopwords.remove('nor')
stopwords.remove('not')

#converting the sentiment column which of type object to integer to perform machine Learning algorithms
df.head(10)

[nltk_data] Downloading package stopwords to /home/kolya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,review,sentiment
0,"Another demonstration of Kurosawa's genius, hi...",1
1,A new side to the story of Victoria and Albert...,1
2,Whoever says pokemon is stupid can die. This m...,1
3,"With Iphigenia, Mikhali Cacoyannis is perhaps ...",1
4,I have to start saying it has been a long time...,1
5,In a time when Hollywood is making money by sh...,1
6,I can't believe how many people hate Hal Spark...,1
7,An hilariously accurate caricature of trying t...,1
8,This is one of the best Fred Astaire-Ginger Ro...,1
9,"Set during WWII, Bedknobs and Broomsticks is a...",1


In [8]:
processed_reviews=[]
for i in tqdm(df['review']):
    #Regular expression that removes all the html tags pressent in the reviews
    i=re.sub('(<[\w\s]*/?>)',"",i)
    #Expanding all the contractions present in the review to is respective actual form
    i=contractions.fix(i)
    #Removing all the special charactesrs from the review text
    i=re.sub('[^a-zA-Z0-9\s]+',"",i)
    #Removing all the digits present in the review text
    
    i=re.sub('\d+',"",i)
    #Making all the review text to be of lower case as well as removing the stopwords and words of length less than 3
    #processed_reviews.append(" ".join([lemmatizer.lemmatize(j.lower()) for j in i.split() if j not in stopwords and len(j)>=3]))
    processed_reviews.append(" ".join([j.lower() for j in i.split() if j not in stopwords and len(j)>=3]))

<>:4: SyntaxWarning: invalid escape sequence '\w'
<>:8: SyntaxWarning: invalid escape sequence '\s'
<>:11: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\w'
<>:8: SyntaxWarning: invalid escape sequence '\s'
<>:11: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_29942/3464384947.py:4: SyntaxWarning: invalid escape sequence '\w'
  i=re.sub('(<[\w\s]*/?>)',"",i)
/tmp/ipykernel_29942/3464384947.py:8: SyntaxWarning: invalid escape sequence '\s'
  i=re.sub('[^a-zA-Z0-9\s]+',"",i)
/tmp/ipykernel_29942/3464384947.py:11: SyntaxWarning: invalid escape sequence '\d'
  i=re.sub('\d+',"",i)
100%|███████████████████████████████████| 25000/25000 [00:06<00:00, 3762.95it/s]


In [9]:
#Creating a new datafram using the Processed Reviews
processed_df=pd.DataFrame({'review':processed_reviews,'sentiment':list(df['sentiment'])})

In [10]:
processed_df.head()

,review,sentiment
0,another demonstration kurosawas genius first c...,1
1,new side story victoria albert brought life di...,1
2,whoever says pokemon stupid die this movie sup...,1
3,with iphigenia mikhali cacoyannis perhaps firs...,1
4,start saying long time since seen seen times w...,1


In [11]:
processed_reviews=[]
for i in tqdm(df_test['review']):
    #Regular expression that removes all the html tags pressent in the reviews
    i=re.sub('(<[\w\s]*/?>)',"",i)
    #Expanding all the contractions present in the review to is respective actual form
    i=contractions.fix(i)
    #Removing all the special charactesrs from the review text
    i=re.sub('[^a-zA-Z0-9\s]+',"",i)
    #Removing all the digits present in the review text
    
    i=re.sub('\d+',"",i)
    #Making all the review text to be of lower case as well as removing the stopwords and words of length less than 3
    #processed_reviews.append(" ".join([lemmatizer.lemmatize(j.lower()) for j in i.split() if j not in stopwords and len(j)>=3]))
    processed_reviews.append(" ".join([j.lower() for j in i.split() if j not in stopwords and len(j)>=3]))

<>:4: SyntaxWarning: invalid escape sequence '\w'
<>:8: SyntaxWarning: invalid escape sequence '\s'
<>:11: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\w'
<>:8: SyntaxWarning: invalid escape sequence '\s'
<>:11: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_29942/3105351429.py:4: SyntaxWarning: invalid escape sequence '\w'
  i=re.sub('(<[\w\s]*/?>)',"",i)
/tmp/ipykernel_29942/3105351429.py:8: SyntaxWarning: invalid escape sequence '\s'
  i=re.sub('[^a-zA-Z0-9\s]+',"",i)
/tmp/ipykernel_29942/3105351429.py:11: SyntaxWarning: invalid escape sequence '\d'
  i=re.sub('\d+',"",i)
100%|███████████████████████████████████| 25000/25000 [00:06<00:00, 3864.05it/s]


In [12]:
processed_df_test=pd.DataFrame({'review':processed_reviews,'sentiment':list(df_test['sentiment'])})

In [13]:
processed_df_test.head()

,review,sentiment
0,nuovomondo great experience many filmmakers te...,1
1,andy lau lau chingwan superb johnny tos tautly...,1
2,superbly crafted lowbudget thriller twists tur...,1
3,first made aware film saw preview another movi...,1
4,the bothersome man smart surreal movie makes r...,1


# <center> Data Preparaton</center>

In [14]:
#Splitting the data into dependent and independent variables i.e, features and the target columns
x_train=processed_df['review']
y_train=processed_df['sentiment']
x_test=processed_df['review']
y_test=processed_df['sentiment']

In [15]:
from gensim.models import Word2Vec

In [16]:
#Preparing data for training the Word2Vec model. It requies each review to be as a list of words.
words_in_sentences=[]
for i in tqdm(x_train):
    words_in_sentences.append(i.split())

100%|█████████████████████████████████| 25000/25000 [00:00<00:00, 103093.16it/s]


In [17]:
print("Model Training Started...")
model = Word2Vec(sentences=words_in_sentences, vector_size=200,workers=-1)
print("Model Training Completed...")

Model Training Started...
Model Training Completed...


In [18]:
#Word2Vec model returns the similar words for a given word
model.wv.most_similar('interesting', topn=10)

[('reef', 0.29141926765441895),
 ('uninhibited', 0.27358847856521606),
 ('bonuses', 0.26930731534957886),
 ('economics', 0.26257264614105225),
 ('seeley', 0.25532135367393494),
 ('shunned', 0.2552047073841095),
 ('knox', 0.2530529499053955),
 ('fifties', 0.25236502289772034),
 ('noel', 0.25053635239601135),
 ('fran', 0.24743491411209106)]

In [19]:
model.wv.similar_by_word('interesting')

[('reef', 0.29141926765441895),
 ('uninhibited', 0.27358847856521606),
 ('bonuses', 0.26930731534957886),
 ('economics', 0.26257264614105225),
 ('seeley', 0.25532135367393494),
 ('shunned', 0.2552047073841095),
 ('knox', 0.2530529499053955),
 ('fifties', 0.25236502289772034),
 ('noel', 0.25053635239601135),
 ('fran', 0.24743491411209106)]

In [20]:
#Word emebedding for a given word.
model.wv.get_vector('interesting')

array([ 4.6840911e-03,  3.7987691e-03, -1.1177796e-03,  1.5590722e-03,
        1.1348677e-03,  1.9045722e-03, -1.7822022e-03,  3.2423241e-03,
        7.2837592e-04, -3.4230137e-03, -4.5624333e-03, -4.0916638e-03,
       -4.0422322e-04, -1.2635291e-04,  1.4038336e-03, -4.1578640e-03,
        1.7171830e-03,  1.5580666e-03,  2.2419810e-03, -4.2805905e-03,
       -4.6675606e-03, -9.5208170e-04, -2.2630095e-03,  4.7974708e-03,
        5.4157973e-04, -3.2363879e-03,  3.2201551e-03,  2.3567141e-03,
        3.2443083e-03, -1.5812754e-04,  4.4327081e-04,  2.8509456e-03,
       -3.7684834e-03,  8.7455811e-04, -1.3972074e-03,  1.4741415e-03,
        1.5198052e-03,  1.4181697e-03, -1.6910321e-03, -4.1781771e-03,
        7.9987169e-04, -1.3889819e-03,  1.9826002e-03,  4.9126870e-03,
        1.6973513e-03,  4.9867537e-03,  3.5538268e-03,  4.1563688e-03,
       -2.2453857e-03,  2.8424858e-04,  1.7713904e-03,  4.5019304e-03,
        2.6141780e-03,  1.0521912e-03,  2.7190344e-03, -1.7987072e-04,
      

In [21]:
model.wv.n_similarity(['king','male'],['queen','female'])

-0.07687268

In [22]:
model.wv.distance('king','queen')

0.9814928416162729

In [23]:
model.wv.doesnt_match(["king", "george","stephen","truck"])

'king'

In [24]:
vocab=list(model.wv.key_to_index.keys())
print(len(vocab))

30410


## Метод усреднения векторов в предложении

In [25]:
def avg_w2vec(sentences):
    """
    This Function is using Average Word2Vec approach for creating a numerical vector for a given review from the word embeddings of each words of the review.
    """
    transformed=[]
    for sentence in tqdm(sentences):
        count=0
        vector=np.zeros(200)
        for word in sentence.split():
            if word in vocab:
                vector+=model.wv.get_vector(word)
                count+=1
        if count!=0:
            vector/=count
            transformed.append(vector)
    return np.array(transformed)

In [26]:
x_train_transformed=avg_w2vec(x_train)
x_test_transformed=avg_w2vec(x_test)

100%|████████████████████████████████████| 25000/25000 [02:30<00:00, 166.00it/s]


# <center> Обучение разных моделей</center>

In [28]:
from sklearn.model_selection import KFold
# KFold с 5 разбиениями
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Функция для проведения валидации
def validate_model(model, X, y, kf, metric):
    mae_scores = []
    
    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        
        # Обучение модели
        model.fit(X_train, y_train)
        
        # Предсказание на тестовом наборе
        y_pred = model.predict(X_test)
        
        mae = metric(y_test, y_pred)
        mae_scores.append(mae)
        
    return mae_scores

In [30]:
# Импорт необходимых библиотек
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Словарь моделей классификации
models_classification = {
    'Logistic Regression': LogisticRegression(),
    'Ridge classifier': RidgeClassifier(), 
    'Ridge alpha = 3': RidgeClassifier(5),
    #'Support Vector Classifier': SVC(max_iter = 1000),
    'KNN': KNeighborsClassifier(),
    'Random Forest Classifier': RandomForestClassifier(n_estimators=50, max_depth=10)
}


In [34]:
x_train_transformed.shape

(25000, 200)

In [36]:
for model_name, model in models_classification.items():
    scores = validate_model(model, x_train_transformed, y_train, kf, accuracy_score)
    print(f"{model_name}: Точность на каждом разбиении: {scores}")
    print(f"{model_name}: Средняя точность: {np.mean(scores)}\n")

Logistic Regression: Точность на каждом разбиении: [0.497, 0.4954, 0.4984, 0.4984, 0.4924]
Logistic Regression: Средняя точность: 0.49632

Ridge classifier: Точность на каждом разбиении: [0.6592, 0.6334, 0.6446, 0.6444, 0.5822]
Ridge classifier: Средняя точность: 0.63276

Ridge alpha = 3: Точность на каждом разбиении: [0.5078, 0.4966, 0.576, 0.5876, 0.493]
Ridge alpha = 3: Средняя точность: 0.5322

KNN: Точность на каждом разбиении: [0.6268, 0.6096, 0.6096, 0.6114, 0.6366]
KNN: Средняя точность: 0.6188

Random Forest Classifier: Точность на каждом разбиении: [0.6558, 0.6552, 0.6624, 0.6426, 0.6598]
Random Forest Classifier: Средняя точность: 0.65516

